<a href="https://colab.research.google.com/github/VICO-27/CSEC-DS-Summer/blob/main/GeoAI_FEATURE_ENGINEERING%20Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================================
# CELL 1: IMPORTS + LOAD DATA
# ==========================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import random

import matplotlib.pyplot as plt
import seaborn as sns

import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score, roc_auc_score, make_scorer
from sklearn.feature_selection import mutual_info_classif
import joblib

plt.style.use("ggplot")
pd.set_option("display.max_columns", None)

train = pd.read_csv("/content/drive/MyDrive/Train.csv")
test = pd.read_csv("/content/drive/MyDrive/Test.csv")

print("Train:", train.shape, " Test:", test.shape)

Train: (1821, 146)  Test: (1030, 145)


In [ ]:
# ==========================================================
# CELL 2: CONSTANTS
# ==========================================================

MONTHS = [f"{i:02d}" for i in range(1, 13)]
RADAR_BANDS = ["VH", "VV"]
OPTICAL_BANDS = ["blue", "green", "red", "re1", "re2", "re3", "nir", "nira", "swir1", "swir2"]
ALL_BANDS = RADAR_BANDS + OPTICAL_BANDS
INDEX_FAMILIES = ["NDVI", "NDWI", "MNDWI", "EVI", "SAVI", "NDBI", "BSI"]
RATIO_FAMILIES = ["VH_VV_ratio", "VH_minus_VV"]
AGG_FAMILIES = ALL_BANDS + INDEX_FAMILIES + RATIO_FAMILIES
EPS = 1e-6

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def zindi_metric(y_true, y_prob):
    y_pred = (y_prob >= 0.5).astype(int)
    return 0.6 * f1_score(y_true, y_pred) + 0.4 * roc_auc_score(y_true, y_prob)

zindi_scorer = make_scorer(zindi_metric, response_method="predict_proba")

print("Constants ready.")

Constants ready.


In [ ]:
# ==========================================================
# CELL 3: FEATURE ENGINEERING PIPELINE (single source of truth)
# Key fix vs v1: row-relative imputation, NOT population median —
# prevents bias toward the majority class for rows with fewer valid months
# ==========================================================

def clean_missing(df):
    df = df.copy()
    feature_cols = [c for c in df.columns if c not in ["ID", "label"]]
    df[feature_cols] = df[feature_cols].replace(-9999, np.nan)
    return df

def row_relative_impute(df, bands):
    df = df.copy()
    for band in bands:
        cols = [f"{band}_{m}" for m in MONTHS]
        row_means = df[cols].mean(axis=1, skipna=True)
        for c in cols:
            df[c] = df[c].fillna(row_means)
        still_missing = df[cols].isnull().any(axis=1)
        if still_missing.any():
            fallback = df[cols].stack().median()
            df[cols] = df[cols].fillna(fallback)
    return df

def compute_indices(df):
    df = df.copy()
    for m in MONTHS:
        nir, red = df[f"nir_{m}"], df[f"red_{m}"]
        blue, green = df[f"blue_{m}"], df[f"green_{m}"]
        swir1 = df[f"swir1_{m}"]
        vh, vv = df[f"VH_{m}"], df[f"VV_{m}"]

        df[f"NDVI_{m}"] = (nir - red) / (nir + red + EPS)
        df[f"NDWI_{m}"] = (green - nir) / (green + nir + EPS)
        df[f"MNDWI_{m}"] = (green - swir1) / (green + swir1 + EPS)
        df[f"EVI_{m}"] = (2.5 * (nir - red)) / (nir + 6*red - 7.5*blue + 1 + EPS)
        df[f"SAVI_{m}"] = (1.5 * (nir - red)) / (nir + red + 0.5 + EPS)
        df[f"NDBI_{m}"] = (swir1 - nir) / (swir1 + nir + EPS)
        df[f"BSI_{m}"] = (swir1 + red - nir - blue) / (swir1 + red + nir + blue + EPS)
        df[f"VH_VV_ratio_{m}"] = vh / (vv + EPS)
        df[f"VH_minus_VV_{m}"] = vh - vv
    return df

def aggregate_temporal(df, features):
    df = df.copy()
    for feature in features:
        cols = [f"{feature}_{m}" for m in MONTHS]
        values = df[cols]
        df[f"{feature}_mean"] = values.mean(axis=1)
        df[f"{feature}_std"] = values.std(axis=1)
        df[f"{feature}_min"] = values.min(axis=1)
        df[f"{feature}_max"] = values.max(axis=1)
        df[f"{feature}_median"] = values.median(axis=1)
        df[f"{feature}_range"] = df[f"{feature}_max"] - df[f"{feature}_min"]
        df[f"{feature}_cv"] = df[f"{feature}_std"] / (df[f"{feature}_mean"].abs() + EPS)
        q1, q3 = values.quantile(0.25, axis=1), values.quantile(0.75, axis=1)
        df[f"{feature}_iqr"] = q3 - q1
    return df

def temporal_dynamics(df, features):
    df = df.copy()
    months_arr = np.arange(1, 13)
    for feature in features:
        cols = [f"{feature}_{m}" for m in MONTHS]
        data = df[cols].values
        df[f"{feature}_peak_month"] = np.argmax(data, axis=1) + 1
        df[f"{feature}_low_month"] = np.argmin(data, axis=1) + 1
        df[f"{feature}_trend"] = [np.polyfit(months_arr, row, 1)[0] for row in data]
    return df

def seasonal_features(df, features):
    df = df.copy()
    wet = ["06", "07", "08", "09"]
    dry = ["01", "02", "03", "04", "05", "10", "11", "12"]
    for feature in features:
        df[f"{feature}_wet_mean"] = df[[f"{feature}_{m}" for m in wet]].mean(axis=1)
        df[f"{feature}_dry_mean"] = df[[f"{feature}_{m}" for m in dry]].mean(axis=1)
        df[f"{feature}_season_diff"] = df[f"{feature}_wet_mean"] - df[f"{feature}_dry_mean"]
    return df

def month_to_month(df, features):
    df = df.copy()
    for feature in features:
        monthly = df[[f"{feature}_{m}" for m in MONTHS]]
        diff = monthly.diff(axis=1)
        df[f"{feature}_largest_rise"] = diff.max(axis=1)
        df[f"{feature}_largest_drop"] = diff.min(axis=1)
        df[f"{feature}_change_std"] = diff.std(axis=1)
    return df

def engineer_features(df):
    """Full pipeline, identical for train and test."""
    df = clean_missing(df)
    df = row_relative_impute(df, ALL_BANDS)
    df = compute_indices(df)
    df = aggregate_temporal(df, AGG_FAMILIES)
    df = temporal_dynamics(df, INDEX_FAMILIES)
    df = seasonal_features(df, INDEX_FAMILIES)
    df = month_to_month(df, ["NDVI", "NDWI", "MNDWI", "EVI", "SAVI"])
    return df

print("Pipeline functions defined.")

Pipeline functions defined.


In [ ]:
# ==========================================================
# CELL 4: APPLY PIPELINE + VERIFY TRAIN/TEST CONSISTENCY
# ==========================================================

train_fe = engineer_features(train)
test_fe = engineer_features(test)

print("train_fe:", train_fe.shape, " test_fe:", test_fe.shape)
print("Column mismatch:", (set(train_fe.columns) - {"label"}) ^ set(test_fe.columns))
print("Remaining NaNs — train:", train_fe.isnull().sum().sum(), " test:", test_fe.isnull().sum().sum())

train_fe: (1821, 479)  test_fe: (1030, 478)
Column mismatch: set()
Remaining NaNs — train: 0  test: 0


In [ ]:
# ==========================================================
# CELL 5: BASELINE FEATURE SELECTION (variance + correlation)
# ==========================================================

from sklearn.feature_selection import VarianceThreshold

X = train_fe.drop(columns=["ID", "label"])
y = train_fe["label"]

selector = VarianceThreshold(threshold=0.0)
selector.fit(X)
X = X[X.columns[selector.get_support()]]

corr = X.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
to_drop = [c for c in upper.columns if any(upper[c] > 0.98)]
X = X.drop(columns=to_drop)

kept_columns = X.columns.tolist()
train_clean = pd.concat([train_fe[["ID", "label"]], train_fe[kept_columns]], axis=1)
test_clean = pd.concat([test_fe[["ID"]], test_fe[kept_columns]], axis=1)

print(f"After variance + correlation pruning: {len(kept_columns)} features")
print("train_clean:", train_clean.shape, " test_clean:", test_clean.shape)

After variance + correlation pruning: 369 features
train_clean: (1821, 371)  test_clean: (1030, 370)


In [ ]:
# ==========================================================
# CELL 6: PREDICTIVE IMPORTANCE (RF + mutual info, combined rank)
# ==========================================================

from sklearn.ensemble import RandomForestClassifier

X = train_clean.drop(columns=["ID", "label"])
y = train_clean["label"]

rf = RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1)
rf.fit(X, y)
rf_imp = pd.Series(rf.feature_importances_, index=X.columns)

mi = mutual_info_classif(X, y, random_state=42, n_jobs=-1)
mi_imp = pd.Series(mi, index=X.columns)

importance_df = pd.DataFrame({"rf_importance": rf_imp, "mutual_info": mi_imp})
importance_df["avg_rank"] = (
    importance_df["rf_importance"].rank(ascending=False) +
    importance_df["mutual_info"].rank(ascending=False)
) / 2
importance_df = importance_df.sort_values("avg_rank")

top_features_raw = importance_df.head(60).index.tolist()
print("Top 15 by predictive rank:")
print(importance_df.head(15))

Top 15 by predictive rank:
               rf_importance  mutual_info  avg_rank
NDWI_07             0.033774     0.403644       3.0
NDVI_07             0.027835     0.403974       3.5
NDWI_04             0.041677     0.392079       4.5
NDVI_wet_mean       0.029674     0.383598       7.5
NDWI_08             0.021043     0.399890       8.0
NDVI_min            0.025596     0.386313       8.5
nir_08              0.023122     0.387558       9.0
nir_07              0.014721     0.405625      10.5
MNDWI_07            0.019928     0.381080      13.0
NDWI_09             0.019951     0.362746      15.0
NDVI_08             0.015998     0.368574      15.5
MNDWI_08            0.020428     0.360862      16.0
swir1_06            0.009863     0.412977      16.5
VV_mean             0.012339     0.373462      18.0
NDVI_04             0.021805     0.343181      20.5


In [ ]:
# ==========================================================
# CELL 7: ADVERSARIAL VALIDATION — detect train/test shift
# High AUC = features leak dataset identity = shift risk
# ==========================================================

adv_data = pd.concat([
    train_clean[top_features_raw].assign(is_test=0),
    test_clean[top_features_raw].assign(is_test=1)
], axis=0).reset_index(drop=True)

X_adv, y_adv = adv_data[top_features_raw], adv_data["is_test"]

adv_auc = cross_val_score(
    lgb.LGBMClassifier(n_estimators=300, random_state=42, n_jobs=-1, verbose=-1),
    X_adv, y_adv, cv=cv, scoring="roc_auc"
)
print(f"Adversarial validation AUC: {adv_auc.mean():.4f} (0.5=safe, 1.0=severe shift)")

adv_model = lgb.LGBMClassifier(n_estimators=300, random_state=42, n_jobs=-1, verbose=-1)
adv_model.fit(X_adv, y_adv)
adv_danger_rank = pd.Series(adv_model.feature_importances_, index=top_features_raw).rank(ascending=False)

print("\nMost dataset-discriminative (dangerous) features:")
print(adv_danger_rank.sort_values().head(10))

Adversarial validation AUC: 0.9990 (0.5=safe, 1.0=severe shift)

Most dataset-discriminative (dangerous) features:
swir2_median        1.0
swir1_cv            2.0
NDVI_cv             3.0
VH_min              4.0
NDVI_min            5.0
VV_mean             6.0
nir_07              7.0
MNDWI_change_std    8.0
nir_06              9.5
swir1_median        9.5
dtype: float64


In [ ]:
# ==========================================================
# CELL 8: BUILD SHIFT-ROBUST FEATURE SET
# Keep only features that are predictive AND not strong dataset discriminators
# ==========================================================

combined = pd.DataFrame({
    "predictive_rank": importance_df.loc[top_features_raw, "avg_rank"],
    "danger_rank": adv_danger_rank,
})
combined["safety_score"] = combined["predictive_rank"] - combined["danger_rank"]
combined = combined.sort_values("safety_score")

robust_features = combined[combined["safety_score"] <= -8.0].index.tolist()

print(f"Shift-robust feature set: {len(robust_features)} features")
print(robust_features)

Shift-robust feature set: 18 features
['NDVI_07', 'NDWI_07', 'NDWI_08', 'NDVI_wet_mean', 'NDWI_04', 'MNDWI_07', 'swir2_mean', 'MNDWI_04', 'NDWI_06', 'NDVI_08', 'MNDWI_08', 'MNDWI_mean', 'NDVI_06', 'NDWI_09', 'NDVI_mean', 'swir2_08', 'nir_08', 'MNDWI_05']


In [ ]:
# ==========================================================
# CELL 9: ADVERSARIAL SAMPLE WEIGHTING
# Train rows that "look like test" get more weight during training
# ==========================================================

adv_final = lgb.LGBMClassifier(n_estimators=300, random_state=42, n_jobs=-1, verbose=-1)
adv_final.fit(
    pd.concat([train_clean[robust_features], test_clean[robust_features]], axis=0),
    pd.concat([pd.Series([0]*len(train_clean)), pd.Series([1]*len(test_clean))], axis=0)
)

train_test_likeness = adv_final.predict_proba(train_clean[robust_features])[:, 1]
sample_weights = train_test_likeness + 0.05
sample_weights = sample_weights / sample_weights.mean()

print("Sample weight stats:")
print(pd.Series(sample_weights).describe())

Sample weight stats:
count    1821.000000
mean        1.000000
std         0.003361
min         0.998287
25%         0.998293
50%         0.998395
75%         0.999929
max         1.023515
dtype: float64


In [ ]:
# ==========================================================
# CELL 10: HYPERPARAMETER TUNING ON ROBUST FEATURES + WEIGHTS
# ==========================================================

from sklearn.model_selection import RandomizedSearchCV

X = train_clean[robust_features]
y = train_clean["label"]

param_dist = {
    "n_estimators": [200, 300, 500],
    "learning_rate": [0.02, 0.05, 0.08],
    "num_leaves": [7, 15, 31],
    "max_depth": [3, 4, 6, -1],
    "min_child_samples": [10, 20, 30],
    "subsample": [0.7, 0.85, 1.0],
    "colsample_bytree": [0.7, 0.85, 1.0],
}

search = RandomizedSearchCV(
    lgb.LGBMClassifier(class_weight="balanced", random_state=42, n_jobs=-1, verbose=-1),
    param_distributions=param_dist, n_iter=40, scoring=zindi_scorer,
    cv=cv, random_state=42, n_jobs=-1, verbose=1, error_score='raise'
)
search.fit(X, y, sample_weight=sample_weights)

print("Best CV Zindi score:", search.best_score_)
print("Best params:", search.best_params_)
print("\nNote: this CV score will look optimistic — it's still measured on full-year")
print("train rows, not test's shifted/partial-year rows. Treat it as relative, not absolute.")

Fitting 5 folds for each of 40 candidates, totalling 200 fits
Best CV Zindi score: 0.9825257185199497
Best params: {'subsample': 0.7, 'num_leaves': 31, 'n_estimators': 500, 'min_child_samples': 30, 'max_depth': 6, 'learning_rate': 0.08, 'colsample_bytree': 0.85}

Note: this CV score will look optimistic — it's still measured on full-year
train rows, not test's shifted/partial-year rows. Treat it as relative, not absolute.


In [ ]:
# ==========================================================
# CELL 11: TRAIN FINAL MODEL + GENERATE SUBMISSION
# ==========================================================

final_model = lgb.LGBMClassifier(
    **search.best_params_, class_weight="balanced", random_state=42, n_jobs=-1, verbose=-1
)
final_model.fit(train_clean[robust_features], train_clean["label"], sample_weight=sample_weights)

test_probs = final_model.predict_proba(test_clean[robust_features])[:, 1]
test_preds = (test_probs >= 0.5).astype(int)

submission = pd.DataFrame({
    "ID": test_clean["ID"],
    "TargetF1": test_preds,
    "TargetRAUC": test_probs
})
submission.to_csv("submission_v5.csv", index=False)

print(submission.head(10))
print("\nPrediction distribution:")
print(submission["TargetF1"].value_counts(normalize=True))

                   ID  TargetF1    TargetRAUC
0  ID_TS_NEW_SBZAYD5I         1  9.999960e-01
1  ID_TS_NEW_7SPRN3PB         1  9.996535e-01
2  ID_TS_NEW_LZOWPHLC         1  9.999976e-01
3  ID_TS_NEW_DN6TUF64         1  9.999996e-01
4  ID_TS_NEW_95N82M49         0  9.616035e-05
5  ID_TS_NEW_BFEAVD50         0  1.243574e-03
6  ID_TS_NEW_D1E33N0Z         1  9.999960e-01
7  ID_TS_NEW_C85XZKDC         0  3.239926e-07
8  ID_TS_NEW_1Y90ZQUH         1  9.969940e-01
9  ID_TS_NEW_MZJ5GJYB         0  4.359480e-06

Prediction distribution:
TargetF1
1    0.568932
0    0.431068
Name: proportion, dtype: float64


In [ ]:
# ==========================================================
# CELL 12: SAVE ARTIFACTS
# ==========================================================

joblib.dump(final_model, "lightgbm_pond_classifier_v5.pkl")
joblib.dump(robust_features, "robust_feature_list.pkl")
train_clean.to_csv("train_cleaned_engineered.csv", index=False)
test_clean.to_csv("test_cleaned_engineered.csv", index=False)

print("Saved model, feature list, and cleaned datasets.")
print(f"Features used: {len(robust_features)}")
print(f"Best hyperparameters: {search.best_params_}")

Saved model, feature list, and cleaned datasets.
Features used: 18
Best hyperparameters: {'subsample': 0.7, 'num_leaves': 31, 'n_estimators': 500, 'min_child_samples': 30, 'max_depth': 6, 'learning_rate': 0.08, 'colsample_bytree': 0.85}


In [ ]:
# ==========================================================
# CELL 13: PER-DATASET QUANTILE NORMALIZATION
# Fit a separate transform on train and on test — each dataset gets
# mapped to the SAME reference distribution using its OWN quantiles.
# This removes absolute calibration shift while preserving each row's
# relative position within its own dataset (which is what matters for
# separating ponds from everything else).
# ==========================================================

from sklearn.preprocessing import QuantileTransformer

# Work with train_clean / test_clean's 369 shared engineered features
feature_cols = [c for c in train_clean.columns if c not in ["ID", "label"]]

def quantile_normalize_independently(train_df, test_df, cols):
    train_out = train_df.copy()
    test_out = test_df.copy()

    for col in cols:
        qt_train = QuantileTransformer(output_distribution="normal", random_state=42)
        qt_test = QuantileTransformer(output_distribution="normal", random_state=42)

        train_out[col] = qt_train.fit_transform(train_df[[col]]).ravel()
        test_out[col] = qt_test.fit_transform(test_df[[col]]).ravel()

    return train_out, test_out

train_norm, test_norm = quantile_normalize_independently(train_clean, test_clean, feature_cols)

print("Normalized train:", train_norm.shape, " test:", test_norm.shape)
print("\nExample — NDVI_07 before vs after (train):")
print(pd.DataFrame({
    "raw": train_clean["NDVI_07"].head(),
    "normalized": train_norm["NDVI_07"].head()
}))

Normalized train: (1821, 371)  test: (1030, 370)

Example — NDVI_07 before vs after (train):
        raw  normalized
0 -0.143932   -1.874584
1  0.641135    2.978200
2 -0.036201   -0.615814
3  0.405024    1.022881
4 -0.192515   -2.447496


In [ ]:
# ==========================================================
# CELL 14: RE-CHECK ADVERSARIAL AUC ON NORMALIZED FEATURES
# This is the real test — if normalization worked, AUC should
# collapse from 0.999 toward 0.5
# ==========================================================

X_adv_norm = pd.concat([
    train_norm[feature_cols].assign(is_test=0),
    test_norm[feature_cols].assign(is_test=1)
], axis=0).reset_index(drop=True)

adv_auc_norm = cross_val_score(
    lgb.LGBMClassifier(n_estimators=300, random_state=42, n_jobs=-1, verbose=-1),
    X_adv_norm[feature_cols], X_adv_norm["is_test"], cv=cv, scoring="roc_auc"
)

print(f"Adversarial AUC AFTER normalization: {adv_auc_norm.mean():.4f}")
print(f"(Before normalization it was 0.9990 — this tells us how much shift is left)")

Adversarial AUC AFTER normalization: 1.0000
(Before normalization it was 0.9990 — this tells us how much shift is left)


In [ ]:
# ==========================================================
# CELL 15: REVERT NORMALIZATION — go back to train_clean/test_clean
# Confirm we're back to the 0.999 baseline before proceeding
# ==========================================================

print("Reverting to train_clean / test_clean (pre-normalization).")
print("train_clean:", train_clean.shape, " test_clean:", test_clean.shape)

Reverting to train_clean / test_clean (pre-normalization).
train_clean: (1821, 371)  test_clean: (1030, 370)


In [ ]:
# ==========================================================
# CELL 16: DISCOVER LATENT REGIONS IN TRAIN VIA CLUSTERING
# Use the RAW yearly-mean bands (most calibration-sensitive, most likely
# to encode "which region/sensor batch") to find 2 natural clusters —
# testing the hypothesis that train itself spans both pilot regions
# ==========================================================

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

region_signal_cols = [f"{b}_mean" for b in ALL_BANDS if f"{b}_mean" in train_clean.columns]
print("Using these columns to detect regional structure:", region_signal_cols)

X_region = train_clean[region_signal_cols]
X_region_scaled = StandardScaler().fit_transform(X_region)

kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
train_clean["region_cluster"] = kmeans.fit_predict(X_region_scaled)

print("\nCluster sizes:")
print(train_clean["region_cluster"].value_counts())

print("\nDoes label distribution differ by cluster? (checking for confound)")
print(train_clean.groupby("region_cluster")["label"].mean())

# Also project test data onto the same clusters — which region does test resemble more?
X_test_region = StandardScaler().fit(X_region).transform(test_clean[region_signal_cols])
test_cluster_pred = kmeans.predict(X_test_region)
print("\nWhich cluster does TEST data resemble (via nearest centroid)?")
print(pd.Series(test_cluster_pred).value_counts(normalize=True))

Using these columns to detect regional structure: ['VH_mean', 'VV_mean', 'blue_mean', 'green_mean', 'red_mean', 're1_mean', 're2_mean', 'swir1_mean', 'swir2_mean']

Cluster sizes:
region_cluster
0    958
1    863
Name: count, dtype: int64

Does label distribution differ by cluster? (checking for confound)
region_cluster
0    0.678497
1    0.098494
Name: label, dtype: float64

Which cluster does TEST data resemble (via nearest centroid)?
0    0.609709
1    0.390291
Name: proportion, dtype: float64


In [ ]:
# ==========================================================
# CELL 17: LEAVE-ONE-REGION-OUT VALIDATION
# Train on cluster 0, validate on cluster 1 -- and vice versa.
# This is the most honest local proxy we can build for the real
# train (region A/time X) -> test (region B/time Y) gap.
# ==========================================================

region_scores = []

for held_out in [0, 1]:
    tr_mask = train_clean["region_cluster"] != held_out
    val_mask = train_clean["region_cluster"] == held_out

    X_tr = train_clean.loc[tr_mask, top_features_raw]
    y_tr = train_clean.loc[tr_mask, "label"]
    X_val = train_clean.loc[val_mask, top_features_raw]
    y_val = train_clean.loc[val_mask, "label"]

    m = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, class_weight="balanced",
                            random_state=42, n_jobs=-1, verbose=-1)
    m.fit(X_tr, y_tr)
    probs = m.predict_proba(X_val)[:, 1]
    preds = (probs >= 0.5).astype(int)

    score = 0.6 * f1_score(y_val, preds) + 0.4 * roc_auc_score(y_val, probs)
    f1 = f1_score(y_val, preds)
    auc = roc_auc_score(y_val, probs)

    region_scores.append({"held_out_cluster": held_out, "n_val": val_mask.sum(),
                           "zindi_score": score, "f1": f1, "auc": auc})
    print(f"Held out cluster {held_out} (n={val_mask.sum()}): Zindi={score:.4f} F1={f1:.4f} AUC={auc:.4f}")

region_df = pd.DataFrame(region_scores)
print("\n" + "="*60)
print(f"LEAVE-ONE-REGION-OUT MEAN: {region_df['zindi_score'].mean():.4f}")
print(f"Compare: random K-fold was ~0.985-0.99, actual LB is ~0.80")
print("="*60)

Held out cluster 0 (n=958): Zindi=0.8655 F1=0.8328 AUC=0.9145
Held out cluster 1 (n=863): Zindi=0.9239 F1=0.8917 AUC=0.9721

LEAVE-ONE-REGION-OUT MEAN: 0.8947
Compare: random K-fold was ~0.985-0.99, actual LB is ~0.80


In [ ]:
# ==========================================================
# CELL 18: RE-CLUSTER USING LABEL-WEAK SIGNALS ONLY
# Radar (VH/VV) showed heavy class overlap in original EDA (~70% overlap),
# so clustering on radar-only should reflect region/sensor, not class
# ==========================================================

weak_signal_cols = ["VH_mean", "VV_mean", "VH_std", "VV_std", "VH_cv", "VV_cv"]
weak_signal_cols = [c for c in weak_signal_cols if c in train_clean.columns]
print("Clustering on:", weak_signal_cols)

X_weak = StandardScaler().fit_transform(train_clean[weak_signal_cols])

kmeans_v2 = KMeans(n_clusters=2, random_state=42, n_init=10)
train_clean["region_cluster_v2"] = kmeans_v2.fit_predict(X_weak)

print("\nCluster sizes:")
print(train_clean["region_cluster_v2"].value_counts())

print("\nLabel balance by cluster (want this CLOSE to equal — confirms no label leakage):")
print(train_clean.groupby("region_cluster_v2")["label"].mean())

# Where does test project onto these clusters?
scaler_weak = StandardScaler().fit(train_clean[weak_signal_cols])
test_weak_proj = kmeans_v2.predict(scaler_weak.transform(test_clean[weak_signal_cols]))
print("\nTest data cluster assignment:")
print(pd.Series(test_weak_proj).value_counts(normalize=True))

Clustering on: ['VH_mean', 'VV_mean', 'VH_std', 'VV_std', 'VH_cv', 'VV_cv']

Cluster sizes:
region_cluster_v2
0    1262
1     559
Name: count, dtype: int64

Label balance by cluster (want this CLOSE to equal — confirms no label leakage):
region_cluster_v2
0    0.245642
1    0.760286
Name: label, dtype: float64

Test data cluster assignment:
0    0.992233
1    0.007767
Name: proportion, dtype: float64


In [ ]:
# ==========================================================
# CELL 19: IF CLUSTERS ARE LABEL-BALANCED, REPEAT LEAVE-ONE-REGION-OUT
# ==========================================================

if train_clean.groupby("region_cluster_v2")["label"].mean().max() < 0.55 and \
   train_clean.groupby("region_cluster_v2")["label"].mean().min() > 0.25:
    print("Clusters look reasonably label-balanced — proceeding with LORO test.\n")

    region_scores_v2 = []
    for held_out in [0, 1]:
        tr_mask = train_clean["region_cluster_v2"] != held_out
        val_mask = train_clean["region_cluster_v2"] == held_out

        X_tr = train_clean.loc[tr_mask, top_features_raw]
        y_tr = train_clean.loc[tr_mask, "label"]
        X_val = train_clean.loc[val_mask, top_features_raw]
        y_val = train_clean.loc[val_mask, "label"]

        m = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, class_weight="balanced",
                                random_state=42, n_jobs=-1, verbose=-1)
        m.fit(X_tr, y_tr)
        probs = m.predict_proba(X_val)[:, 1]
        preds = (probs >= 0.5).astype(int)

        score = 0.6 * f1_score(y_val, preds) + 0.4 * roc_auc_score(y_val, probs)
        region_scores_v2.append({"held_out": held_out, "n": val_mask.sum(), "zindi_score": score})
        print(f"Held out cluster {held_out} (n={val_mask.sum()}): Zindi={score:.4f}")

    print(f"\nMEAN (label-balanced LORO): {pd.DataFrame(region_scores_v2)['zindi_score'].mean():.4f}")
else:
    print("Clusters are STILL confounded with label (imbalanced). Radar signal alone")
    print("isn't separating region from class either — see note below cell output.")

Clusters are STILL confounded with label (imbalanced). Radar signal alone
isn't separating region from class either — see note below cell output.


In [ ]:
# ==========================================================
# CELL 20: FINAL MODEL — conservative regularization,
# physically-grounded features (water indices + robust aggregates only,
# dropping the data-mined single-month picks that showed up as
# adversarially "dangerous" earlier)
# ==========================================================

final_feature_set = [
    "NDWI_mean", "NDWI_wet_mean", "NDWI_dry_mean", "NDWI_season_diff",
    "MNDWI_mean", "MNDWI_wet_mean", "MNDWI_dry_mean", "MNDWI_season_diff",
    "NDVI_mean", "NDVI_wet_mean", "NDVI_dry_mean", "NDVI_season_diff",
    "swir1_mean", "swir2_mean", "green_mean",
    "NDWI_07", "NDVI_07", "MNDWI_07",
]
final_feature_set = [f for f in final_feature_set if f in train_clean.columns]
print(f"Final conservative feature set: {len(final_feature_set)} features")
print(final_feature_set)

conservative_model = lgb.LGBMClassifier(
    n_estimators=200, learning_rate=0.03, max_depth=4, num_leaves=15,
    min_child_samples=40, subsample=0.7, colsample_bytree=0.7,
    reg_alpha=1.0, reg_lambda=1.0,
    class_weight="balanced", random_state=42, n_jobs=-1, verbose=-1
)

cv_check = cross_val_score(conservative_model, train_clean[final_feature_set], train_clean["label"],
                            cv=cv, scoring=zindi_scorer)
print(f"\nConservative model CV score: {cv_check.mean():.4f} (± {cv_check.std():.4f})")
print("(Expect this to be LOWER than our earlier 0.98s — that's the point. Less overfit = more honest.)")

Final conservative feature set: 14 features
['NDWI_dry_mean', 'NDWI_season_diff', 'MNDWI_mean', 'MNDWI_season_diff', 'NDVI_mean', 'NDVI_wet_mean', 'NDVI_dry_mean', 'NDVI_season_diff', 'swir1_mean', 'swir2_mean', 'green_mean', 'NDWI_07', 'NDVI_07', 'MNDWI_07']

Conservative model CV score: 0.9814 (± 0.0100)
(Expect this to be LOWER than our earlier 0.98s — that's the point. Less overfit = more honest.)


In [ ]:
# ==========================================================
# CELL 21: TRAIN ON FULL DATA + GENERATE FINAL SUBMISSION
# ==========================================================

conservative_model.fit(train_clean[final_feature_set], train_clean["label"])

test_probs_final = conservative_model.predict_proba(test_clean[final_feature_set])[:, 1]
test_preds_final = (test_probs_final >= 0.5).astype(int)

submission_final = pd.DataFrame({
    "ID": test_clean["ID"],
    "TargetF1": test_preds_final,
    "TargetRAUC": test_probs_final
})
submission_final.to_csv("submission_final_conservative.csv", index=False)

print(submission_final.head(10))
print("\nPrediction distribution:")
print(submission_final["TargetF1"].value_counts(normalize=True))
print("\n(Note: competition brief says test may have a HIGHER pond rate than train's 40% —")
print(" so if this lands around 45-55%, that's plausible and NOT necessarily a bug.)")

                   ID  TargetF1  TargetRAUC
0  ID_TS_NEW_SBZAYD5I         1    0.981716
1  ID_TS_NEW_7SPRN3PB         1    0.975376
2  ID_TS_NEW_LZOWPHLC         1    0.923964
3  ID_TS_NEW_DN6TUF64         1    0.989266
4  ID_TS_NEW_95N82M49         0    0.029046
5  ID_TS_NEW_BFEAVD50         1    0.516885
6  ID_TS_NEW_D1E33N0Z         1    0.982148
7  ID_TS_NEW_C85XZKDC         0    0.008815
8  ID_TS_NEW_1Y90ZQUH         1    0.710423
9  ID_TS_NEW_MZJ5GJYB         0    0.068834

Prediction distribution:
TargetF1
1    0.569903
0    0.430097
Name: proportion, dtype: float64

(Note: competition brief says test may have a HIGHER pond rate than train's 40% —
 so if this lands around 45-55%, that's plausible and NOT necessarily a bug.)


In [ ]:
# ==========================================================
# CELL 22: SAVE FINAL ARTIFACTS FOR REPRODUCIBILITY
# ==========================================================

joblib.dump(conservative_model, "final_model.pkl")
joblib.dump(final_feature_set, "final_feature_set.pkl")
train_clean.to_csv("train_cleaned_final.csv", index=False)
test_clean.to_csv("test_cleaned_final.csv", index=False)

print("Saved: final_model.pkl, final_feature_set.pkl, train_cleaned_final.csv, test_cleaned_final.csv")

Saved: final_model.pkl, final_feature_set.pkl, train_cleaned_final.csv, test_cleaned_final.csv
